In [1]:
import os
import numpy as np
import pandas as pd
%pip install scipy
from scipy.stats import chisquare

Note: you may need to restart the kernel to use updated packages.


In [2]:
no_corr_beginner_not_correct = 35
no_corr_beginner_correct = 19
no_corr_expert_not_correct = 14+1
no_corr_expert_correct = 7


#       No correlation            'Correct'                  'Not correct'  
data = [
    ['Beginner', 'No correlation', no_corr_beginner_correct, no_corr_beginner_not_correct],
    ['Expert', 'No correlation',   no_corr_expert_correct,   no_corr_expert_not_correct],
]

df = pd.DataFrame(data, columns=['Experience', 'Correlation Type', 'Correct', 'Not correct'])
df





,Experience,Correlation Type,Correct,Not correct
0,Beginner,No correlation,19,35
1,Expert,No correlation,7,15


In [3]:
df_long = df.melt(
    id_vars=['Experience', 'Correlation Type'],
    value_vars=['Correct', 'Not correct'],
    var_name='Proxy identified',
    value_name='Observed'
)

df_long['Expected'] = df_long.groupby(['Experience'])['Observed'].transform('sum') * df_long.groupby('Proxy identified')['Observed'].transform('sum') / df_long['Observed'].sum()
df_long['Chi-Square'] = (df_long['Observed'] - df_long['Expected']) ** 2 / df_long['Expected']
df_long['p-value'] = df_long.apply(lambda row: chisquare([row['Observed'], row['Expected']])[1], axis=1)

df_long

/tmp/ipykernel_35756/2320410683.py:8: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_long['Expected'] = df_long.groupby(['Experience'])['Observed'].transform('sum') * df_long.groupby('Proxy identified')['Observed'].transform('sum') / df_lo

,Experience,Correlation Type,Proxy identified,Observed,Expected,Chi-Square,p-value
0,Beginner,No correlation,Correct,19,18.473684,0.014995,0.931485
1,Expert,No correlation,Correct,7,7.526316,0.036805,0.890168
2,Beginner,No correlation,Not correct,35,35.526316,0.007797,0.950028
3,Expert,No correlation,Not correct,15,14.473684,0.019139,0.922769


In [4]:
scenario_data = {
    "No correlation": {1: 0, 2: 0, 3: 19, 4: 35, 5: 0},
    "No correlation (R)":       {1: 0, 2: 0, 3: 7, 4: 14, 5: 1},

    "Negative correlation": {1: 1, 2: 12, 3: 14, 4: 22, 5: 10},
    "Negative correlation (R)":       {1: 2, 2: 6, 3: 6, 4: 10, 5: 2},

    "Moderate positive correlation": {1: 0, 2: 2, 3: 5, 4: 37, 5: 12},
    "Moderate positive correlation (R)":       {1: 0, 2: 0, 3: 3, 4: 17, 5: 7},

    "Strong positive correlation": {1: 0, 2: 2, 3: 6, 4: 47, 5: 29},
    "Strong positive correlation (R)":       {1: 0, 2: 0, 3: 1, 4: 16, 5: 10}, 
}